# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
%pip -q install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

from dotenv import load_dotenv

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Cannot locate project root; open the notebook from the project folder.")

PROJECT_ROOT = find_project_root(Path.cwd())
load_dotenv(PROJECT_ROOT / ".env", override=True)
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
COREF_PROVIDER = get_secret("COREF_PROVIDER", "openai").lower()
COREF_MODEL = get_secret("COREF_MODEL", "gpt-4o-mini")
COREF_BATCH_SIZE = int(get_secret("COREF_BATCH_SIZE", "12"))
COREF_MAX_WORKERS = int(get_secret("COREF_MAX_WORKERS", "5"))
EXTRACTION_PROVIDER = get_secret("EXTRACTION_PROVIDER", "openai").lower()
EXTRACTION_MODEL = get_secret("EXTRACTION_MODEL", "gpt-4o-mini")
EXTRACTION_BATCH_SIZE = int(get_secret("EXTRACTION_BATCH_SIZE", "10"))
EXTRACTION_MAX_WORKERS = int(get_secret("EXTRACTION_MAX_WORKERS", "5"))
GENERATION_PROVIDER = get_secret("GENERATION_PROVIDER", "openai").lower()
GENERATION_MODEL = get_secret("GENERATION_MODEL", "gpt-4o-mini")
HF_TOKEN = get_secret("HF_TOKEN", "")
GRAPH_SCOPE = get_secret("GRAPH_SCOPE", "day19-hackernoon-v1")

DATA_PATH = str(OUTPUT_DIR / "hackernoon_subset.csv")
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [3]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = str(OUTPUT_DIR / "hackernoon_subset.csv")
DOWNLOAD_TMP = f"{OUTPUT_CSV}.part"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 1_000_000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True
REUSE_EXISTING = (
    Path(OUTPUT_CSV).exists()
    and os.path.getsize(OUTPUT_CSV) >= LIMIT_MB * 0.95 * 1024 * 1024
)

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Kiểm tra CSV cục bộ..." if REUSE_EXISTING else "Đang kết nối luồng dữ liệu (streaming)...")

try:
    if REUSE_EXISTING:
        with open(OUTPUT_CSV, encoding="utf-8", newline="") as existing_file:
            dataset = [next(csv.DictReader(existing_file))]
    else:
        dataset = load_dataset(
            DATASET_NAME,
            split="train",
            streaming=True,
            token=HF_TOKEN,
        )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(DOWNLOAD_TMP, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(DOWNLOAD_TMP) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(DOWNLOAD_TMP) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(DOWNLOAD_TMP) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    if REUSE_EXISTING:
        os.remove(DOWNLOAD_TMP)
        final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
        with open(OUTPUT_CSV, encoding="utf-8", errors="ignore") as count_file:
            rows_written = sum(1 for _ in count_file) - 1
    else:
        final_size_mb = os.path.getsize(DOWNLOAD_TMP) / (1024 * 1024)
        os.replace(DOWNLOAD_TMP, OUTPUT_CSV)
    print(
        f"✅ {'Tái sử dụng CSV hợp lệ' if REUSE_EXISTING else 'Hoàn thành tải'}: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Kiểm tra CSV cục bộ...
Đang ghi dữ liệu vào: D:\AIIA\K4-Track3-Lab19-GraphRAG\outputs\hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]

✅ Tái sử dụng CSV hợp lệ: D:\AIIA\K4-Track3-Lab19-GraphRAG\outputs\hackernoon_subset.csv
   Rows: 514,417
   Size: 300.00 MB


In [4]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX entity_graph_scope IF NOT EXISTS
        FOR (n:Entity) ON (n.graph_scope)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.


✅ Schema ready.


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

GOLDEN_WORKBOOK_PATH = PROJECT_ROOT / "data" / "graphrag_golden_50_first5000_detailed.xlsx"
if not GOLDEN_WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Missing Golden Dataset workbook: {GOLDEN_WORKBOOK_PATH}")
golden_source_df = pd.read_excel(GOLDEN_WORKBOOK_PATH, sheet_name="Golden Tests")
GOLDEN_EVIDENCE_ROW_IDS = {
    int(row_id)
    for value in golden_source_df["evidence_row_ids_0based"]
    for row_id in json.loads(value)
}
print(f"Golden evidence rows available: {len(GOLDEN_EVIDENCE_ROW_IDS)}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["source_row_index"] = raw.index.astype(int)
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    df["is_golden_evidence"] = df["source_row_index"].isin(GOLDEN_EVIDENCE_ROW_IDS)
    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        golden_rows = df[df["is_golden_evidence"]].copy()
        remaining = df[~df["is_golden_evidence"]]
        sample_n = max(0, LAB_MAX_ARTICLES - len(golden_rows))
        sampled = remaining.sample(min(sample_n, len(remaining)), random_state=SEED)
        df = pd.concat([golden_rows, sampled], ignore_index=True)
        df = df.sort_values("source_row_index").reset_index(drop=True)
    print(
        f"Lab sample: {len(df):,} articles; "
        f"golden evidence preserved={int(df['is_golden_evidence'].sum())}"
    )
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "source_row_index": int(r.source_row_index),
                "is_golden_evidence": bool(r.is_golden_evidence),
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

Golden evidence rows available: 28

Exact dedup: 245,324 -> 212,212
Lab sample: 1,500 articles; golden evidence preserved=28


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,source_row_index,is_golden_evidence,text
0,8c5930949a3d9f3c3a38::c0000,8c5930949a3d9f3c3a38,Samsung Showcases Groundbreaking Logic Innovations at System LSI Tech Day 2023,2023-10-05,43,True,Samsung Electronics Co. Ltd. a world leader in advanced semiconductor technology today unveiled its latest innovatio...
1,566e41efa663f190cf11::c0000,566e41efa663f190cf11,L&T Technology Services and Qualcomm Selected by Thales for Enabling 5G Private Networks in Urban Railways,2023-02-23,261,True,L&T Technology Services Limited (BSE ... sales offices and 91 innovation labs as of December 31 2022. For more infor...
2,1906ed4e6e32c255c7af::c0000,1906ed4e6e32c255c7af,Keysight and Synopsys Partner for IoT Device Cybersecurity,2023-09-21,272,True,The global IoT device market is experiencing notable growth due to the rise in adoption of IoT devices and is projec...
3,67f61078101e32548302::c0000,67f61078101e32548302,AP Open AI agree to share select news content and technology in new collaboration,2023-07-13,366,True,The Associated Press and OpenAI have reached an agreement to share access to select news content and technology as t...
4,4a67a8e832498753f8f3::c0000,4a67a8e832498753f8f3,16 Leading Technology and Service Providers Launch Industry's First SASE Product and Services Certification,2023-10-03,369,False,MEF a global industry association of network cloud security and technology providers accelerating enterprise digital...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
from openai import OpenAI
groq_client = (
    Groq(api_key=GROQ_API_KEY, max_retries=0, timeout=60.0)
    if GROQ_API_KEY else None
)
openai_client = (
    OpenAI(api_key=OPENAI_API_KEY, max_retries=2, timeout=45.0)
    if OPENAI_API_KEY else None
)

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def retry_delay_seconds(error, attempt):
    response = getattr(error, "response", None)
    headers = getattr(response, "headers", {}) or {}
    retry_after = headers.get("retry-after")
    try:
        return min(60.0, max(1.0, float(retry_after)))
    except (TypeError, ValueError):
        return min(60.0, 2 ** attempt + random.random())

def is_retryable_error(error):
    response = getattr(error, "response", None)
    status = getattr(error, "status_code", None) or getattr(response, "status_code", None)
    try:
        return status is None or int(status) in {408, 409, 429, 500, 502, 503, 504}
    except (TypeError, ValueError):
        return True

def groq_chat(messages, model=None, json_mode=False, max_retries=3):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
                "max_completion_tokens": 1200,
            }
            if model.startswith("openai/gpt-oss-"):
                kwargs["reasoning_effort"] = "low"
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1 or not is_retryable_error(e):
                break
            time.sleep(retry_delay_seconds(e, attempt))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

def openai_chat(messages, model=None, json_mode=False):
    if openai_client is None:
        raise RuntimeError("Thiếu OPENAI_API_KEY.")
    model = model or COREF_MODEL
    if not model:
        raise RuntimeError("Thiếu COREF_MODEL.")
    kwargs = {
        "model": model,
        "messages": messages,
        "temperature": 0.0,
        "max_completion_tokens": 3000,
    }
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = openai_client.chat.completions.create(**kwargs)
    usage = {}
    if getattr(resp, "usage", None):
        usage = {
            "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
            "completion_tokens": getattr(resp.usage, "completion_tokens", None),
            "total_tokens": getattr(resp.usage, "total_tokens", None),
        }
    return resp.choices[0].message.content, usage

def openai_json(system, user, model=None):
    text, usage = openai_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

def generation_chat(messages, json_mode=False):
    if GENERATION_PROVIDER == "openai":
        return openai_chat(messages, model=GENERATION_MODEL, json_mode=json_mode)
    if GENERATION_PROVIDER == "groq":
        return groq_chat(messages, model=GENERATION_MODEL, json_mode=json_mode)
    raise ValueError("GENERATION_PROVIDER must be openai or groq.")

def generation_json(system, user):
    text, usage = generation_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        json_mode=True,
    )
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

from concurrent.futures import ThreadPoolExecutor, as_completed

COREF_CUE_RE = re.compile(
    r"\b(?:he|him|his|she|her|hers|they|them|their|theirs|it|its|former|latter)\b"
    r"|\b(?:this|that|these|those|the)\s+"
    r"(?:company|firm|startup|executive|platform|product|service|technology|organization)\b",
    flags=re.I,
)
COREF_CACHE_PATH = OUTPUT_DIR / (
    f"coref_v2_{COREF_PROVIDER}_{re.sub(r'[^a-zA-Z0-9._-]+', '_', COREF_MODEL)}.jsonl"
)

def needs_coref(text):
    return bool(COREF_CUE_RE.search(norm_space(text)))

def coref_json(system, user):
    if COREF_PROVIDER == "openai":
        return openai_json(system, user, model=COREF_MODEL)
    if COREF_PROVIDER == "groq":
        return groq_json(system, user, model=COREF_MODEL)
    raise ValueError("COREF_PROVIDER must be openai or groq.")

def load_coref_cache():
    if not COREF_CACHE_PATH.exists():
        return pd.DataFrame(columns=["chunk_id", "resolved_text", "unresolved_mentions"])
    try:
        cached = pd.read_json(COREF_CACHE_PATH, lines=True, dtype={"chunk_id": str})
        required = {"chunk_id", "resolved_text", "unresolved_mentions"}
        if not required.issubset(cached.columns):
            raise ValueError("invalid cache columns")
        return cached[list(required)]
    except Exception as e:
        print(f"Cảnh báo: bỏ qua coref cache không hợp lệ: {e}")
        return pd.DataFrame(columns=["chunk_id", "resolved_text", "unresolved_mentions"])

def save_coref_cache(cache_df):
    tmp_path = COREF_CACHE_PATH.with_suffix(COREF_CACHE_PATH.suffix + ".tmp")
    cache_df.drop_duplicates("chunk_id", keep="last").to_json(
        tmp_path, orient="records", lines=True, force_ascii=False
    )
    os.replace(tmp_path, COREF_CACHE_PATH)

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = coref_json(COREF_SYSTEM, prompt)
    items = obj.get("items")
    if not isinstance(items, list) or not all(isinstance(x, dict) for x in items):
        raise ValueError("Coreference response must contain an items list of objects.")
    returned_ids = [x.get("chunk_id") for x in items]
    expected_ids = batch_df["chunk_id"].astype(str).tolist()
    if len(returned_ids) != len(set(returned_ids)) or not set(returned_ids).issubset(set(expected_ids)):
        raise ValueError(f"Coreference response contains duplicate or unknown chunk_id: returned={returned_ids}")
    # Missing items mean the model found no safe replacement; preserve the original text.
    by_id = {x["chunk_id"]: x for x in items}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        unresolved = item.get("unresolved_mentions", [])
        if not isinstance(unresolved, list):
            unresolved = [norm_space(unresolved)] if norm_space(unresolved) else []
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": unresolved,
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=COREF_BATCH_SIZE, max_workers=COREF_MAX_WORKERS):
    if chunks_subset.empty:
        raise ValueError("Cannot run coreference on an empty chunk set.")
    if batch_size <= 0 or max_workers <= 0:
        raise ValueError("batch_size and max_workers must be positive.")

    source = chunks_subset.copy()
    source["chunk_id"] = source["chunk_id"].astype(str)
    if source["chunk_id"].duplicated().any():
        raise ValueError("chunk_id must be unique before coreference.")

    wanted_ids = set(source["chunk_id"])
    cached = load_coref_cache()
    cached["chunk_id"] = cached["chunk_id"].astype(str)
    cached = cached[cached["chunk_id"].isin(wanted_ids)].drop_duplicates("chunk_id", keep="last")
    result_by_id = {r["chunk_id"]: r for r in cached.to_dict("records")}

    cue_mask = source["text"].map(needs_coref)
    for row in source.loc[~cue_mask].itertuples(index=False):
        result_by_id[row.chunk_id] = {
            "chunk_id": row.chunk_id,
            "resolved_text": row.text,
            "unresolved_mentions": [],
        }

    pending = source[cue_mask & ~source["chunk_id"].isin(result_by_id)].copy()
    batches = [
        (start, pending.iloc[start:start + batch_size].copy())
        for start in range(0, len(pending), batch_size)
    ]
    print(
        f"Coref {COREF_PROVIDER}:{COREF_MODEL} | tổng={len(source)}, "
        f"cache={len(cached)}, bỏ qua={int((~cue_mask).sum())}, "
        f"gửi LLM={len(pending)} ({len(batches)} batches, {max_workers} workers)"
    )

    errors = []
    cache_df = cached.copy()
    if batches:
        with ThreadPoolExecutor(max_workers=min(max_workers, len(batches))) as pool:
            future_map = {pool.submit(resolve_coref_batch, batch): (start, batch) for start, batch in batches}
            for future in tqdm(as_completed(future_map), total=len(future_map), desc="Coref API batches"):
                start, batch = future_map[future]
                try:
                    df, _ = future.result()
                    for row in df.to_dict("records"):
                        result_by_id[str(row["chunk_id"])] = row
                    cache_df = pd.concat([cache_df, df], ignore_index=True)
                    save_coref_cache(cache_df)
                except Exception as e:
                    errors.append({
                        "start": int(start),
                        "end": int(start + len(batch) - 1),
                        "chunk_ids": ",".join(batch["chunk_id"].astype(str)),
                        "error_type": type(e).__name__,
                        "error": str(e),
                    })
                    for row in batch.itertuples(index=False):
                        result_by_id[row.chunk_id] = {
                            "chunk_id": row.chunk_id,
                            "resolved_text": row.text,
                            "unresolved_mentions": ["COREF_BATCH_FAILED"],
                        }

    ordered = [result_by_id[chunk_id] for chunk_id in source["chunk_id"]]
    return pd.DataFrame(ordered), pd.DataFrame(errors)

golden_chunks = chunks_df[chunks_df["is_golden_evidence"]].copy()
non_golden_chunks = chunks_df[~chunks_df["is_golden_evidence"]].copy()
remaining_slots = max(0, EXTRACTION_MAX_CHUNKS - len(golden_chunks))
extraction_source = pd.concat(
    [golden_chunks, non_golden_chunks.head(remaining_slots)], ignore_index=True
).head(EXTRACTION_MAX_CHUNKS)
print(
    f"Extraction scope: {len(extraction_source)} chunks; "
    f"golden evidence chunks={int(extraction_source['is_golden_evidence'].sum())}"
)
coref_df, coref_errors_df = run_coref(
    extraction_source, batch_size=COREF_BATCH_SIZE, max_workers=COREF_MAX_WORKERS
)
if not coref_errors_df.empty:
    display(coref_errors_df.head(10))
    raise RuntimeError("Coreference failed for one or more batches; inspect coref_errors_df and retry before Section 2.")
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

Extraction scope: 400 chunks; golden evidence chunks=28
Coref openai:gpt-4o-mini | tổng=400, cache=178, bỏ qua=222, gửi LLM=0 (0 batches, 5 workers)


In [8]:
coref_failures = coref_df["unresolved_mentions"].apply(
    lambda x: "COREF_BATCH_FAILED" in x
).sum()

print("Coreference batch failures:", coref_failures)

Coreference batch failures: 0


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [9]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extraction_json(system, user):
    if EXTRACTION_PROVIDER == "openai":
        return openai_json(system, user, model=EXTRACTION_MODEL)
    if EXTRACTION_PROVIDER == "groq":
        return groq_json(system, user, model=EXTRACTION_MODEL)
    raise ValueError("EXTRACTION_PROVIDER must be openai or groq.")

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return exactly one item for every input chunk_id, using an empty relations list when no supported fact exists.
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return extraction_json(EXTRACT_SYSTEM, prompt)

def safe_confidence(value):
    try:
        return max(0.0, min(1.0, float(str(value).replace(",", "."))))
    except (TypeError, ValueError):
        return 0.0

def run_extraction(source_df, batch_size=EXTRACTION_BATCH_SIZE, max_workers=EXTRACTION_MAX_WORKERS):
    required = {"chunk_id", "published_date", "text", "resolved_text"}
    if source_df.empty or not required.issubset(source_df.columns):
        raise ValueError(f"Extraction source is empty or missing columns: {sorted(required - set(source_df.columns))}")
    if batch_size <= 0 or max_workers <= 0:
        raise ValueError("batch_size and max_workers must be positive.")
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    def process_batch(start, batch):
        obj, _ = extract_batch(batch)
        items = obj.get("items")
        if not isinstance(items, list) or not all(isinstance(x, dict) for x in items):
            raise ValueError("Extraction response must contain an items list of objects.")
        returned_ids = [str(x.get("chunk_id")) for x in items]
        expected_ids = batch["chunk_id"].astype(str).tolist()
        if len(returned_ids) != len(set(returned_ids)) or not set(returned_ids).issubset(set(expected_ids)):
            raise ValueError(f"Extraction response contains duplicate or unknown chunk_id: returned={returned_ids}")
        # Missing items are valid zero-relation chunks; never fabricate triples.

        batch_triples = []
        for item in items:
            cid = str(item.get("chunk_id"))
            relations = item.get("relations", [])
            if not isinstance(relations, list):
                continue
            for x in relations:
                if not isinstance(x, dict):
                    continue
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                evidence = norm_space(x.get("evidence"))
                if (not s or not t or not evidence or st not in ALLOWED_NODE_TYPES
                        or tt not in ALLOWED_NODE_TYPES or rel not in ALLOWED_RELATIONS):
                    continue
                batch_triples.append({
                    "source_raw": s, "source_type": st, "relation": rel,
                    "target_raw": t, "target_type": tt,
                    "source_chunk_id": cid, "published_date": meta[cid] or "",
                    "evidence": evidence, "confidence": safe_confidence(x.get("confidence")),
                })
        return batch_triples

    jobs = [(start, source_df.iloc[start:start + batch_size].copy())
            for start in range(0, len(source_df), batch_size)]
    print(
        f"NER+RE {EXTRACTION_PROVIDER}:{EXTRACTION_MODEL} | "
        f"chunks={len(source_df)}, batches={len(jobs)}, workers={max_workers}"
    )
    with ThreadPoolExecutor(max_workers=min(max_workers, len(jobs))) as pool:
        future_map = {pool.submit(process_batch, start, batch): (start, batch) for start, batch in jobs}
        for future in tqdm(as_completed(future_map), total=len(future_map), desc="NER+RE API batches"):
            start, batch = future_map[future]
            try:
                triples.extend(future.result())
            except Exception as e:
                errors.append({
                    "start": int(start), "end": int(start + len(batch) - 1),
                    "error_type": type(e).__name__, "error": str(e),
                })

    triple_columns = ["source_raw", "source_type", "relation", "target_raw", "target_type", "source_chunk_id", "published_date", "evidence", "confidence"]
    error_columns = ["start", "end", "error_type", "error"]
    return pd.DataFrame(triples, columns=triple_columns), pd.DataFrame(errors, columns=error_columns)

raw_triples_df, extraction_errors_df = run_extraction(
    extraction_source, batch_size=EXTRACTION_BATCH_SIZE, max_workers=EXTRACTION_MAX_WORKERS
)
if not extraction_errors_df.empty:
    display(extraction_errors_df.head(10))
    raise RuntimeError("Extraction failed for one or more batches; fix the errors before ingestion.")
if raw_triples_df.empty:
    raise RuntimeError("No valid triples were extracted; do not continue to ingestion.")
display(raw_triples_df.head())

NER+RE openai:gpt-4o-mini | chunks=400, batches=40, workers=5


NER+RE API batches:   0%|          | 0/40 [00:00<?, ?it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Dell Technologies Inc.,Company,USES,Microsoft Azure,Technology,e826dd76339fcf0d6a9b::c0000,2023-05-26,The new Apex Cloud Platforms integrates services from Microsoft Azure,1.0
1,Cohere,Company,DEVELOPED,AI tools,Technology,6ec376fffd2c56fef630::c0000,2023-07-26,technology access from the startup Cohere,1.0
2,Cohere,Company,DEVELOPED,AI tools,Technology,b432186d8fe246dc1fab::c0000,2023-07-27,technology access from the startup Cohere,1.0
3,Amazon Web Services,Company,USES,Advanced Micro Devices Inc.,Company,ec3ce18d568e33867724::c0000,2023-06-14,is considering using new artificial intelligence chips from Advanced Micro Devices Inc,1.0
4,ServiceNow,Company,WORKED_AT,Patrick Walravens,Person,db743e3c8c1ead5aa840::c0000,2023-05-18,JMP Securities analyst Patrick Walravens maintained a Buy rating on ServiceNow.,0.9


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [10]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

LEXICAL_MERGE_THRESHOLD = 0.86

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= LEXICAL_MERGE_THRESHOLD

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    required = {"source_type", "source_raw", "target_type", "target_raw"}
    if raw_triples_df.empty or not required.issubset(raw_triples_df.columns):
        raise ValueError("Cannot resolve entities from an empty or invalid triple table.")
    if not 0.0 <= threshold <= 1.0 or top_k <= 0:
        raise ValueError("threshold must be in [0,1] and top_k must be positive.")
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if t == "Company" and norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j:
                    continue
                score = float(score)
                if score < threshold:
                    if score >= 0.50:
                        audit.append({
                            "type": typ, "left": names[i], "right": names[j],
                            "similarity": score, "decision": "REJECT_BELOW_THRESHOLD"
                        })
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": score,
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    if raw_df.empty:
        raise ValueError("Cannot canonicalize an empty triple table.")
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        manual = MANUAL_ALIASES.get(n) if typ == "Company" else None
        return mapping.get((typ, n), manual or name)

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{GRAPH_SCOPE}:{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{GRAPH_SCOPE}:{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
if triples_df.empty:
    raise RuntimeError("No non-self-loop triples remain after entity resolution.")
display(entity_resolution_audit_df.head(20))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,type,left,right,similarity,decision
0,Company,Dell Technologies Inc.,Dell,0.729939,REJECT_BELOW_THRESHOLD
1,Company,Dell Technologies Inc.,Advanced Micro Devices Inc.,0.651120,REJECT_BELOW_THRESHOLD
2,Company,Dell Technologies Inc.,Verb Technology Company Inc.,0.634944,REJECT_BELOW_THRESHOLD
3,Company,Dell Technologies Inc.,Uber Technologies Inc.,0.621909,REJECT_BELOW_THRESHOLD
4,Company,Advanced Micro Devices Inc.,MicroTech,0.691400,REJECT_BELOW_THRESHOLD
5,Company,Advanced Micro Devices Inc.,Uber Technologies Inc.,0.559842,REJECT_BELOW_THRESHOLD
6,Company,Advanced Micro Devices Inc.,Samsung Electronics Co. Ltd.,0.557068,REJECT_BELOW_THRESHOLD
7,Company,Samsung Electronics Co. Ltd.,Samsung,0.720685,REJECT_BELOW_THRESHOLD
8,Company,Samsung Electronics Co. Ltd.,Reliance Industries Ltd,0.546980,REJECT_BELOW_THRESHOLD
9,Company,Samsung,Apple,0.575256,REJECT_BELOW_THRESHOLD


In [11]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ, "graph_scope":GRAPH_SCOPE,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    if size <= 0:
        raise ValueError("Batch size must be positive.")
    for i in range(0, len(records), size):
        yield records[i:i+size]

def clear_graph_scope():
    existing = run_cypher(
        "MATCH (n:Entity {graph_scope:$scope}) RETURN count(n) AS n", scope=GRAPH_SCOPE
    )[0]["n"]
    run_cypher(
        "MATCH (n:Entity {graph_scope:$scope}) DETACH DELETE n", scope=GRAPH_SCOPE
    )
    print(f"Cleared {existing} existing nodes from graph_scope={GRAPH_SCOPE!r}.")

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.graph_scope=row.graph_scope,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        RETURN count(n) AS merged
        """
        for b in batches(part.to_dict("records"), batch_size):
            result = run_cypher(query, rows=b)
            if not result or int(result[0]["merged"]) != len(b):
                raise RuntimeError(f"Node ingestion mismatch for {typ}: expected {len(b)}, got {result}")

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_id", "target_id", "relation", "source_chunk_id", "published_date", "evidence", "confidence"}
    if not required.issubset(triples_df.columns):
        raise ValueError(f"Missing edge columns: {sorted(required - set(triples_df.columns))}")
    for col in ["source_id", "target_id", "source_chunk_id", "published_date", "evidence"]:
        if triples_df[col].fillna("").astype(str).str.strip().eq("").any():
            raise ValueError(f"Blank required edge field: {col}")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{
            source_chunk_id: row.source_chunk_id,
            evidence_hash: row.evidence_hash
        }}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence,
            r.graph_scope=$scope
        RETURN count(r) AS merged
        """

        rows = part.copy()
        rows["evidence_hash"] = rows["evidence"].map(
            lambda value: sha1(norm_space(value).lower())[:24]
        )
        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","evidence_hash","confidence"]
        for b in batches(rows[cols].to_dict("records"), batch_size):
            result = run_cypher(query, rows=b, scope=GRAPH_SCOPE)
            if not result or int(result[0]["merged"]) != len(b):
                raise RuntimeError(f"Edge ingestion mismatch for {rel}: expected {len(b)}, got {result}")

nodes_df = build_nodes(triples_df)
if nodes_df.empty:
    raise RuntimeError("No nodes were built; do not continue to ingestion.")
clear_graph_scope()
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

Cleared 294 existing nodes from graph_scope='day19-hackernoon-v1'.


In [12]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH (:Entity {graph_scope:$scope})-[r]->(:Entity {graph_scope:$scope})
    WHERE r.graph_scope=$scope AND (
      trim(coalesce(r.source_chunk_id, ''))='' OR
      trim(coalesce(r.published_date, ''))='' OR
      trim(coalesce(r.evidence, ''))=''
    )
    RETURN count(r) AS n
    """, scope=GRAPH_SCOPE)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity {graph_scope:$scope}) RETURN count(n) AS n", scope=GRAPH_SCOPE)[0]["n"],
        "edges": run_cypher("MATCH (:Entity {graph_scope:$scope})-[r]->(:Entity {graph_scope:$scope}) WHERE r.graph_scope=$scope RETURN count(r) AS n", scope=GRAPH_SCOPE)[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity {graph_scope:$scope})
    OPTIONAL MATCH (n)-[r]-(m:Entity {graph_scope:$scope})
    WHERE r IS NULL OR r.graph_scope=$scope
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """, scope=GRAPH_SCOPE))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 289, 'edges': 163, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,24ffd5649c29b4b11419b20e,Adidas,Company,9
1,2a1cfc2bf5f4f1f3d30f0dcf,Microsoft,Company,4
2,46a12a04de5ceb813432e64a,Activision Blizzard,Company,4
3,4716559199ef93187037ff95,Airbnb,Company,3
4,00be1679159e62fa434d8f34,Jurgen Klopp,Person,3
5,77575f11f43a679d0fd4d11c,Advance Auto Parts,Company,3
6,c3f62dc1c54b7908da8598f8,Liverpool FC,Company,2
7,8430761f8cfb01fbc7c04c98,Adocia,Company,2
8,c88b039f36d643403637b633,AbbVie,Company,2
9,05113ae5a9619a149e9319d8,Algerian Ministry of National Defence,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [13]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    if chunks_df.empty or "text" not in chunks_df.columns:
        raise ValueError("Cannot build flat index from an empty or invalid chunk table.")
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    if flat_index is None or flat_store is None:
        raise RuntimeError("Flat index is not built; run build_flat_index(chunks_df) first.")
    if not norm_space(query) or k <= 0:
        return "", pd.DataFrame(columns=["score", "chunk_id", "published_date", "text"])
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [14]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = generation_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    seeds = obj.get("seeds")
    if not isinstance(seeds, list) or not all(isinstance(x, dict) for x in seeds):
        raise ValueError("Seed response must contain a seeds list of objects.")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in seeds
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    if nodes_df.empty or not {"id", "name", "type"}.issubset(nodes_df.columns):
        raise ValueError("Cannot build entity matcher from an empty or invalid node table.")
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.75):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity {graph_scope:$scope})
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"], scope=GRAPH_SCOPE)

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [15]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id, graph_scope:$scope})
    OPTIONAL MATCH (n)-[r]-(m:Entity {graph_scope:$scope})
    WHERE r IS NULL OR r.graph_scope=$scope
    RETURN count(r) AS degree
    """, id=node_id, scope=GRAPH_SCOPE)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id, graph_scope:$scope})
    MATCH (n)-[r]-(m:Entity {graph_scope:$scope})
    WHERE r.graph_scope=$scope
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      r.evidence_hash AS evidence_hash,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit), scope=GRAPH_SCOPE)

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    if not norm_space(query):
        out = {"context": "", "edges": pd.DataFrame(), "diagnostics": {"reason": "EMPTY_QUERY", "supernode_events": []}}
        return out if return_debug else ""
    if max_hops <= 0 or edge_limit <= 0:
        raise ValueError("max_hops and edge_limit must be positive.")
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"], e["relation"], e["target_id"], e["source_chunk_id"], e.get("evidence_hash"))
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [16]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    if not norm_space(context):
        return {"answer": "Insufficient retrieval context.", "latency_s": 0.0, "total_tokens": 0}
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = generation_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}]
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [17]:
#@title 4.1 — Golden Dataset từ workbook có evidence thật
GOLDEN_PATH = str(PROJECT_ROOT / "data" / "golden_dataset.csv")
GOLDEN_SELECTED_IDS = ["G5000-41", "G5000-39", "G5000-49", "G5000-46", "G5000-33"]
GOLDEN_COLUMNS = ["id", "group", "question", "reference_answer", "reference_evidence"]

golden_df = (
    golden_source_df[golden_source_df["id"].isin(GOLDEN_SELECTED_IDS)]
    .set_index("id").loc[GOLDEN_SELECTED_IDS].reset_index()[GOLDEN_COLUMNS]
)
golden_df.to_csv(GOLDEN_PATH, index=False)
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id", "group", "question", "reference_answer", "reference_evidence"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if df.empty or df["id"].astype(str).duplicated().any():
        raise ValueError("Golden Dataset must be non-empty and have unique ids.")
    if df["question"].fillna("").astype(str).str.strip().eq("").any():
        raise ValueError("Golden Dataset contains blank questions.")
    if require_answers and df.reference_answer.fillna("").astype(str).str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    evidence = df["reference_evidence"].fillna("").astype(str).str.strip()
    if require_answers and (evidence.eq("") | evidence.str.contains("TO_BE_FILLED|Validate against", case=False, regex=True)).any():
        raise ValueError("Điền reference_evidence thật trước final evaluation.")
    print("✅ Golden Dataset valid.")

,id,group,question,reference_answer,reference_evidence
0,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ...",Axis Security; a unified Secure Access Service Edge (SASE) solution.,row 4762 (2023-03-02 22:32:00): Hewlett Packard Enterprise Fortifies Network Security With Acquisition of Security S...
1,G5000-39,multi-hop,What two strategic capability areas did HPE expand in 2023 through the Axis Security deal and its later AI cloud ann...,"HPE moved to expand edge-to-cloud security by agreeing to acquire Axis Security, enabling a unified SASE offering. L...",row 4762 (2023-03-02 22:32:00): Hewlett Packard Enterprise Fortifies Network Security With Acquisition of Security S...
2,G5000-49,multi-hop,"Across the selected Samsung records, identify three distinct technology domains Samsung is connected to and the spec...",Display/biometric sensing: Samsung unveiled a Sensor OLED Display with an embedded light-sensing organic photodiode ...,row 472 (2023-05-23 22:48:00): Samsung unveils OLED display with embedded heart rate sensor | row 395 (2023-10-05 23...
3,G5000-46,cross-doc,Distinguish the cybersecurity relationships in the Keysight–Synopsys article and the LTTS–Palo Alto Networks article...,Keysight and Synopsys are the partners in the IoT-device cybersecurity article. L&T Technology Services and Palo Alt...,row 272 (2023-09-21 16:15:00): Keysight and Synopsys Partner for IoT Device Cybersecurity | row 471 (2023-06-30 08:5...
4,G5000-33,cross-doc,"Which July OpenAI-related event is a content/technology collaboration, and which July event is a voluntary governanc...",The AP–OpenAI agreement is a collaboration to share access to select news content and technology for generative-AI u...,row 366 (2023-07-13 00:00:00): AP Open AI agree to share select news content and technology in new collaboration | r...


In [18]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY, max_retries=5, timeout=60.0)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def safe_judge_score(value):
    try:
        return max(1, min(5, int(round(float(value)))))
    except (TypeError, ValueError):
        return 1

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = safe_judge_score(obj.get(k, 1))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [19]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = str(OUTPUT_DIR / "graphrag_eval_checkpoint.csv")

def atomic_to_csv(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

def evaluation_key(row):
    return (str(row.id), norm_space(row.question), norm_space(row.reference_answer))

def run_evaluation(golden_df):
    validate_golden(golden_df, require_answers=True)
    rows, done = [], set()
    if Path(CHECKPOINT).exists():
        checkpoint_df = pd.read_csv(CHECKPOINT)
        key_cols = {"id", "question", "reference_answer"}
        if key_cols.issubset(checkpoint_df.columns):
            rows = checkpoint_df.to_dict("records")
            done = {evaluation_key(r) for r in checkpoint_df.itertuples(index=False)}
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        if evaluation_key(q) in done:
            continue
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        result_df = pd.DataFrame(rows).drop_duplicates(["id", "question", "reference_answer"], keep="last")
        atomic_to_csv(result_df, CHECKPOINT)
    return pd.DataFrame(rows).drop_duplicates(["id", "question", "reference_answer"], keep="last")

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.
✅ Golden Dataset valid.


Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ...",Axis Security; a unified Secure Access Service Edge (SASE) solution.,"Hewlett Packard Enterprise (HPE) agreed to acquire Axis Security, a cloud security provider, to expand its edge-to-c...","Hewlett Packard Enterprise (HPE) agreed to acquire Axis Security, a cloud security provider, to expand its edge-to-c...",5,5,5,5,5,5,1.507954,1.344614,645,509,The candidate accurately identifies the company HPE agreed to acquire (Axis Security) and correctly states that the ...,The candidate accurately identifies the company HPE agreed to acquire (Axis Security) and correctly states the unifi...,0
1,G5000-39,multi-hop,What two strategic capability areas did HPE expand in 2023 through the Axis Security deal and its later AI cloud ann...,"HPE moved to expand edge-to-cloud security by agreeing to acquire Axis Security, enabling a unified SASE offering. L...",Hewlett Packard Enterprise (HPE) expanded its strategic capability areas in 2023 through the Axis Security deal and ...,Hewlett Packard Enterprise (HPE) expanded its strategic capability areas in 2023 through the acquisition of Axis Sec...,3,3,4,4,3,3,2.056602,1.915032,695,729,The candidate provides a general overview of HPE's strategic expansions but lacks specific mention of the two distin...,The candidate provides a general overview of HPE's strategic expansions but lacks specific mention of the two distin...,0
2,G5000-49,multi-hop,"Across the selected Samsung records, identify three distinct technology domains Samsung is connected to and the spec...",Display/biometric sensing: Samsung unveiled a Sensor OLED Display with an embedded light-sensing organic photodiode ...,Samsung is connected to the following three distinct technology domains:\n\n1. **Semiconductor Technology**: Samsung...,Samsung is connected to the following three distinct technology domains:\n\n1. **Smart Home Technology**: Samsung ha...,2,5,2,5,2,5,2.470836,3.464817,725,741,The candidate identifies three technology domains but inaccurately includes Information and Communication Technology...,"The candidate accurately identifies three distinct technology domains: Smart Home Technology, Display Technology, an...",0
3,G5000-46,cross-doc,Distinguish the cybersecurity relationships in the Keysight–Synopsys article and the LTTS–Palo Alto Networks article...,Keysight and Synopsys are the partners in the IoT-device cybersecurity article. L&T Technology Services and Palo Alt...,"The Keysight–Synopsys article focuses on IoT-device cybersecurity, as it discusses the security of software running ...","The Keysight–Synopsys article focuses on IoT-device cybersecurity, as it discusses the growth of the IoT device mark...",5,5,5,5,5,5,2.163997,1.448612,656,507,"The candidate accurately distinguishes between the two articles, correctly identifying the Keysight–Synopsys article...","The candidate accurately distinguishes between the two articles, correctly identifying the Keysight–Synopsys article...",0
4,G5000-33,cross-doc,"Which July OpenAI-related event is a content/technology collaboration, and which July event is a voluntary governanc...",The AP–OpenAI agreement is a collaboration to share access to select news content and technology for generative-AI u...,The July OpenAI-related event that is a content/technology collaboration is the agreement with the Associated Press ...,The July OpenAI-related event that is a content/technology collaboration is the agreement with The Associated Press ...,5,5,5,5,5,5,1.792334,1.963572,657,634,The candidate accurately identifies th

In [20]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    required_cols = {col for pair in metric_map.values() for col in pair} | {"group"}
    if eval_df.empty or not required_cols.issubset(eval_df.columns):
        raise ValueError(f"Evaluation table is empty or missing columns: {sorted(required_cols - set(eval_df.columns))}")

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if pd.isna(f) or pd.isna(gr):
                comment = "Không đủ dữ liệu để so sánh."
            elif metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
atomic_to_csv(eval_results_df, OUTPUT_DIR / "graphrag_eval_results.csv")
atomic_to_csv(comparison_df, OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv")
print("✅ Exported benchmark deliverables to", OUTPUT_DIR)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),1.978,1.706,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,656.500,570.500,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.508,1.345,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,645.000,509.000,GraphRAG không đắt hơn trong sample này.


✅ Exported benchmark deliverables to D:\AIIA\K4-Track3-Lab19-GraphRAG\outputs


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [21]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity {graph_scope:$scope})-[r]-(m:Entity {graph_scope:$scope})
    WHERE r.graph_scope=$scope
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """, scope=GRAPH_SCOPE)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': '24ffd5649c29b4b11419b20e', 'name': 'Adidas', 'degree': 9} fetched= 9


,type,left,right,similarity,decision
48,Company,Affirm Holdings,Affirm Holdings Inc,0.963123,MERGE_VECTOR
20,Company,Liverpool,Liverpool FC,0.908026,REJECT_GUARD
53,Company,Netweb Technologies,Netweb Technologies India,0.896083,REJECT_BELOW_THRESHOLD
42,Company,Adidas,Adidas AG,0.895461,REJECT_BELOW_THRESHOLD
57,Company,Albertsons Cos.,Albertsons,0.879554,REJECT_BELOW_THRESHOLD
56,Company,Kroger Co.,Kroger,0.863680,REJECT_BELOW_THRESHOLD
65,Technology,AI tools,AI program,0.778819,REJECT_BELOW_THRESHOLD
66,Technology,AI tools,AI services,0.764293,REJECT_BELOW_THRESHOLD
69,Technology,artificial intelligence,AI program,0.747773,REJECT_BELOW_THRESHOLD
47,Company,AES El Salvador,AES Andes,0.732794,REJECT_BELOW_THRESHOLD


High-similarity rejected pairs:


,type,left,right,similarity,decision
20,Company,Liverpool,Liverpool FC,0.908026,REJECT_GUARD


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [22]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity {graph_scope:$scope})-[r]->(b:Entity {graph_scope:$scope})
    WHERE r.graph_scope=$scope
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges), scope=GRAPH_SCOPE))
    if edge_df.empty:
        print("Graph has no scoped edges; no communities were built.")
        return pd.DataFrame(columns=["id", "community_id"])

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

print("Bonus community scaffold ready (optional; not required for deliverables).")

Bonus community scaffold ready (optional; not required for deliverables).


In [23]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = generation_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    value = obj.get("sufficient")
    if isinstance(value, bool):
        sufficient = value
    elif str(value).strip().lower() in {"true", "false"}:
        sufficient = str(value).strip().lower() == "true"
    else:
        raise ValueError(f"Invalid sufficient value: {value!r}")
    return sufficient, norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"]) if g2["context"] else (False, "No 2-hop graph context")
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"]) if g3["context"] else (False, "No 3-hop graph context")
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

print("Bonus self-correction scaffold ready (optional; not required for deliverables).")

Bonus self-correction scaffold ready (optional; not required for deliverables).


# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau